In [1]:
import boto3
import json

In [2]:
bedrock = boto3.client(service_name='bedrock-runtime')
lambda_client = boto3.client("lambda")

In [3]:
MODEL_ID = "amazon.nova-pro-v1:0"
MODEL_ID_GPT_OSS_120B= "openai.gpt-oss-120b-1:0"

In [4]:
# Define o toolSpec da função Lambda
# A função Lambda getWeather já foi criada
weather_tool = {
    "toolSpec": {
        "name": "getWeather",
        "description": "Obtém o clima atual a partir da latitude e longitude.",
        "inputSchema": {
            "json": {
                "type": "object",
                "properties": {
                    "lat": {"type": "number"},
                    "lon": {"type": "number"}
                },
                "required": ["lat", "lon"]
            }
        }
    }
}

# Invoca a função Lambda (ferramenta)
def invoke_weather_lambda(input_data):
    response = lambda_client.invoke(
        FunctionName="weather-tool",
        InvocationType="RequestResponse",
        Payload=json.dumps(input_data)
    )
    payload = response["Payload"].read()
    result = json.loads(payload)
    body = result.get("body", "{}")
    return json.loads(body) if isinstance(body, str) else body

In [5]:
initial_user_msg = {
    "role": "user",
    "content": [{"text": "Como está o tempo em Boston?"}]
}

system_prompt=[
    {"text": "Você é um assistente prestativo que pode consultar o clima usando ferramentas quando necessário"}
]

In [6]:
# Faz a pergunta inicial ao modelo
initial_response = bedrock.converse(
    modelId=MODEL_ID,
    system=system_prompt,
    messages=[initial_user_msg],
    toolConfig={
        "tools": [weather_tool],
        "toolChoice": {"auto": {}}
    },
    inferenceConfig={"temperature": 0.7}
)

In [7]:
# Analisa a resposta do modelo para saber se a ferramenta é necessária
assistant_msg = initial_response["output"]["message"]
content_blocks = assistant_msg["content"]
tool_use_block = next((block for block in content_blocks if "toolUse" in block), None)

In [8]:
initial_response

{'ResponseMetadata': {'RequestId': '7bc96207-819b-4cba-8191-2877137c50d2',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Thu, 22 Oct 2026 20:36:36 GMT',
   'content-type': 'application/json',
   'content-length': '578',
   'connection': 'keep-alive',
   'x-amzn-requestid': '7bc96207-819b-4cba-8191-2877137c50d2'},
  'RetryAttempts': 0},
 'output': {'message': {'role': 'assistant',
   'content': [{'text': "<thinking> To find out the current weather in Boston, I need to use the 'getWeather' tool. However, I need the latitude and longitude coordinates for Boston. Once I have these coordinates, I can call the tool to get the weather information. </thinking>\n\n"},
    {'toolUse': {'toolUseId': 'tooluse_1J7wN3SPEUadYsNeJRDk8P',
      'name': 'getWeather',
      'input': {'lon': -71.0589, 'lat': 42.3601}}}]}},
 'stopReason': 'tool_use',
 'usage': {'inputTokens': 432, 'outputTokens': 89, 'totalTokens': 521},
 'metrics': {'latencyMs': 951}}

In [9]:
if not tool_use_block:
    print("=== Resposta do modelo ===")
    print(content_blocks[0]["text"])
else:
    tool_use = tool_use_block["toolUse"]
    tool_input = tool_use["input"]
    tool_use_id = tool_use["toolUseId"]

    print(f"-> O modelo solicitou a ferramenta getWeather com a entrada: {tool_input}")

    # Invoca a ferramenta
    tool_output = invoke_weather_lambda(tool_input)
    print(f"-> A Lambda retornou: {tool_output}")

    # Monta o texto com a saída da ferramenta
    try:
        summary = f"O clima em {tool_output['location']} no momento é {tool_output['temperature']}, com {tool_output['condition']}."
    except Exception as e:
        summary = f"Desculpe, não consegui obter o clima. ({str(e)})"

    # Monta a mensagem com o resultado da ferramenta
    tool_result_msg = {
        "role": "user",
        "content": [{
            "toolResult": {
                "toolUseId": tool_use_id,
                "content": [{"text": summary}]
            }
        }]
    }

    final_response = bedrock.converse(
        modelId=MODEL_ID,
        messages=[initial_user_msg, assistant_msg, tool_result_msg],
        toolConfig={
            "tools": [weather_tool],
            "toolChoice": {"auto": {}}
        },
        inferenceConfig={"temperature": 0.7}
    )

    final_msg = final_response["output"]["message"]["content"][0]["text"]
    print(final_msg)

-> O modelo solicitou a ferramenta getWeather com a entrada: {'lon': -71.0589, 'lat': 42.3601}


ResourceNotFoundException: An error occurred (ResourceNotFoundException) when calling the Invoke operation: Function not found: arn:aws:lambda:us-east-1:<ACCOUNT_ID>:function:weather-tool:$LATEST